In [ ]:
import tensorflow as tf
from tensorflow import keras

In [ ]:
(x_train_data, y_train_data), (x_val_data, y_val_data) = keras.datasets.fashion_mnist.load_data()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
def preprocessing_function(x_new, y_new):
  x_new = tf.cast(x_new, tf.float32) / 255.0  # Normalize pixel values
  y_new = tf.cast(y_new, tf.int64)            # Convert labels to integers
  return x_new, y_new

def func_creating_dataset(xs_data, ys_data, num_classes=10):
  ys_data = tf.one_hot(ys_data, depth=num_classes)  # One-hot encode labels
  return tf.data.Dataset.from_tensor_slices((xs_data, ys_data)) \
    .map(preprocessing_function) \
    .shuffle(buffer_size=1000) \
    .batch(32)

# Create the training and validation datasets using func_creating_dataset
dataset_training = func_creating_dataset(x_train_data, y_train_data) # Creating the training dataset
dataset_val = func_creating_dataset(x_val_data, y_val_data)       # Creating the validation dataset

In [ ]:
My_model = keras.Sequential([
    keras.layers.Reshape(target_shape=(28 * 28,), input_shape=(28, 28)),  # Flatten the input
    keras.layers.Dense(units=256, activation='relu'),                    # Hidden layer 1 : A layer with 256 neurons and the ReLU activation function.
    keras.layers.Dense(units=192, activation='relu'),                    # Hidden layer 2
    keras.layers.Dense(units=128, activation='relu'),                    # Hidden layer 3 : A layer with 128 neurons for more learning power.
    keras.layers.Dense(units=10, activation='softmax')                   # Output layer :  Outputs 10 probabilities (one for each class).
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/reshape.py:39: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
My_model.compile(optimizer='adam',
              loss=tf.losses.CategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
history = My_model.fit(
    dataset_training.repeat(),
    epochs=10,
    steps_per_epoch=500,
    validation_data=dataset_val.repeat(),
    validation_steps=2
)

Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/nn.py:666: UserWarning: "`categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


500/500 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.7000 - loss: 0.8389 - val_accuracy: 0.8281 - val_loss: 0.4870
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.8267 - loss: 0.4892 - val_accuracy: 0.8906 - val_loss: 0.2887
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8511 - loss: 0.4217 - val_accuracy: 0.9219 - val_loss: 0.2885
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.8492 - loss: 0.4121 - val_accuracy: 0.7812 - val_loss: 0.5305
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8577 - loss: 0.3748 - val_accuracy: 0.8750 - val_loss: 0.3894
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8689 - loss: 0.3654 - val_accuracy: 0.8281 - val_loss: 0.4371
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.8768 - loss: 0.3528 - val_accuracy: 0.8125 - val_loss: 0.5317
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8666 - loss: 0.3568 - val_accuracy: 0.8750 - v

In [ ]:
Make_predictions = My_model.predict(dataset_val)
print(Make_predictions)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
[[9.29693878e-01 2.45939409e-05 1.25096156e-03 ... 1.55245434e-08
  4.64681762e-05 4.08923825e-07]
 [4.12663639e-01 2.33034792e-04 1.15261953e-02 ... 5.54322287e-06
  8.34416307e-04 8.04429419e-06]
 [1.35968067e-02 3.77135158e-01 4.22668690e-03 ... 3.18444567e-03
  2.82224640e-02 1.08889155e-01]
 ...
 [2.87822727e-15 4.72800628e-25 1.11120781e-19 ... 5.30966256e-16
  1.10003884e-13 1.56736010e-14]
 [4.43443716e-01 2.30015479e-02 4.89781313e-02 ... 1.44179026e-03
  2.14118101e-02 2.88978824e-03]
 [1.65508054e-02 1.44842476e-01 4.55399463e-03 ... 3.78886034e-04
  3.20076421e-02 3.84466513e-03]]


In [ ]:
test_loss, test_accuracy = My_model.evaluate(dataset_val)
print(f"Test Accuracy: {test_accuracy:.2f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8576 - loss: 0.3908
Test Accuracy: 0.85
